In [ ]:
import sys
from pathlib import Path

# --- Colab/Local Setup ---
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Running in Google Colab. Setting up environment...")
    from google.colab import drive
    drive.mount('/content/drive')
    project_path = "/content/drive/MyDrive/data"
    !ln -sfn {project_path} /project
    BASE_PATH = Path("/project")
    DATASET_PATH = BASE_PATH / "dataset_lma_effort.pkl"
    REPO_ROOT = Path(__file__).resolve().parent.parent if "__file__" in dir() else Path("..").resolve()
    sys.path.append(str(REPO_ROOT))
    %pip install -q bvhio
else:
    print("Not in Colab. Assuming local setup.")
    BASE_PATH = Path("./")
    REPO_ROOT = Path("..").resolve()
    sys.path.insert(0, str(REPO_ROOT))
    DATASET_PATH = REPO_ROOT / "datasets" / "lma_effort.pkl"


In [ ]:
from pathlib import Path

from source.dataset import build_lma_effort_dataset

LMA_EFFORT_DATASET_ROOT = Path('/project/extracted_fingerless')  # update for your environment
DATASET_PATH = LMA_EFFORT_DATASET_ROOT.parent / 'lma_effort.pkl'

dataset = build_lma_effort_dataset(
    dataset_root=LMA_EFFORT_DATASET_ROOT,
    save_path=DATASET_PATH,
    device='cpu',
    sequence_length=50,
    max_frame_diff=1,
)
print(dataset.laban_limits)
del dataset


In [ ]:
# ==============================================================================
# Imports
# ==============================================================================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
from tqdm import tqdm

from source.dataset import MotionDataset
from source.interpolator import Interpolator

# ==============================================================================
# Configuration
# ==============================================================================
class TrainingConfig:
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    DATASET_PATH = DATASET_PATH
    SAVE_MODEL_PATH = BASE_PATH / 'interpolator.pth'
    BATCH_SIZE = 64
    LEARNING_RATE = 1e-4
    NUM_EPOCHS = 800
    VAL_SPLIT = 0.1
    SEQUENCE_LENGTH = 50

    NUM_POINTS = 4
    NUM_COORDS = 3
    LATENT_DIM = 64
    ENCODER_HIDDEN_DIM = 512
    DECODER_HIDDEN_DIM = 512

    W_RECON = 1.0

    # Cyclical beta-annealing (manuscript Section 3.3.1):
    # beta(t) varies between 0.0 and 0.01 over 10 cycles; the first half
    # of each cycle ramps linearly from 0.0 to BETA_MAX and the second
    # half is held at BETA_MAX (Fu et al. 2019).
    BETA_START = 0.0
    BETA_MAX = 0.01
    BETA_NUM_CYCLES = 10
    BETA_RAMP_RATIO = 0.5

    # Style-descriptor noise / randomization during training
    # (manuscript Section 3.3.2): +/- 0.2 uniform noise on V and H,
    # R uniformly randomized in [0, 1].
    NOISE_VH = 0.2

config = TrainingConfig()
print(f"Using device: {config.DEVICE}")
print(f"Model will be saved to: {config.SAVE_MODEL_PATH}")

# ==============================================================================
# Dataset
# ==============================================================================
print(f"Loading pre-processed dataset from {config.DATASET_PATH}...")
if not config.DATASET_PATH.exists():
    raise FileNotFoundError(f"Dataset not found at {config.DATASET_PATH}.")

motion_dataset = MotionDataset.load(config.DATASET_PATH, device='cpu')


class MotionDataWrapper(Dataset):
    def __init__(self, source: MotionDataset):
        self.data = source.training_data

    def __len__(self) -> int:
        return len(self.data['positions_sites'])

    def __getitem__(self, index: int):
        chunk = {key: self.data[key][index] for key in self.data}
        return {
            'full_sequence': chunk['positions_sites'],
            'conditions': torch.stack([
                chunk['laban_v'], chunk['laban_h'],
                chunk['laban_p'], chunk['laban_r'],
            ], dim=-1),
        }


def collate_fn(batch):
    return {
        'full_sequence': torch.stack([item['full_sequence'] for item in batch]),
        'conditions': torch.stack([item['conditions'] for item in batch]),
    }


full_dataset = MotionDataWrapper(motion_dataset)
train_size = int(len(full_dataset) * (1 - config.VAL_SPLIT))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True,
                          collate_fn=collate_fn, pin_memory=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE, shuffle=False,
                        collate_fn=collate_fn, pin_memory=True, num_workers=2)
print(f"Training samples: {len(train_dataset)}, Validation samples: {len(val_dataset)}")


# ==============================================================================
# Model, loss, optimizer, scheduler
# ==============================================================================
model = Interpolator(
    seq_len=config.SEQUENCE_LENGTH,
    num_points=config.NUM_POINTS,
    num_coords=config.NUM_COORDS,
    encoder_hidden_dim=config.ENCODER_HIDDEN_DIM,
    decoder_hidden_dim=config.DECODER_HIDDEN_DIM,
    latent_dim=config.LATENT_DIM,
    num_style_descriptors=4,
).to(config.DEVICE)


class InterpolatorLoss(nn.Module):
    """L_CVAE = lambda_recon * L_recon + beta(t) * L_KLD (manuscript Section 3.3.1).

    L_recon emphasizes endpoint accuracy via L_recon + 10 * L_start + 10 * L_end.
    """

    def __init__(self, w_recon: float, beta: float = 0.0):
        super().__init__()
        self.w_recon = w_recon
        self.beta = beta
        self.mse = nn.MSELoss(reduction='sum')

    def forward(self, recon, x, mu, log_var):
        batch_size = x.size(0)
        l_recon = self.mse(recon, x) / batch_size
        l_start = self.mse(recon[:, 0], x[:, 0]) / batch_size
        l_end = self.mse(recon[:, -1], x[:, -1]) / batch_size
        l_recon = l_recon + 10.0 * l_start + 10.0 * l_end
        kld = -0.5 * torch.mean(torch.sum(1 + log_var - mu.pow(2) - log_var.exp(), dim=1))
        total = self.w_recon * l_recon + self.beta * kld
        return total, {'total': total.item(), 'recon': l_recon.item(), 'kld': kld.item()}


loss_fn = InterpolatorLoss(w_recon=config.W_RECON, beta=0.0)
optimizer = optim.AdamW(model.parameters(), lr=config.LEARNING_RATE, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=15, verbose=True,
)
print(f"Model has {sum(p.numel() for p in model.parameters() if p.requires_grad):,} trainable parameters.")


def cyclical_beta(epoch: int, num_epochs: int, num_cycles: int, beta_max: float, ramp_ratio: float = 0.5) -> float:
    cycle_len = max(1, num_epochs // num_cycles)
    pos_in_cycle = (epoch % cycle_len) / cycle_len
    if pos_in_cycle < ramp_ratio:
        return beta_max * (pos_in_cycle / ramp_ratio)
    return beta_max


def apply_training_noise(conditions: torch.Tensor, noise_vh: float) -> torch.Tensor:
    """+/- noise on V and H, uniform R randomization (Section 3.3.2)."""
    v_noise = (torch.rand_like(conditions[:, 0]) * 2.0 - 1.0) * noise_vh
    h_noise = (torch.rand_like(conditions[:, 1]) * 2.0 - 1.0) * noise_vh
    conditions[:, 0] = conditions[:, 0] + v_noise
    conditions[:, 1] = conditions[:, 1] + h_noise
    conditions[:, 3] = torch.rand_like(conditions[:, 3])
    return conditions


def run_epoch(model, loader, loss_fn, device, is_training, optimizer=None):
    model.train(is_training)
    totals = {'total': 0.0, 'recon': 0.0, 'kld': 0.0}
    pbar = tqdm(loader, desc="Training" if is_training else "Validating", leave=False)
    for batch in pbar:
        x = batch['full_sequence'].to(device)
        conditions = batch['conditions'].to(device).clone()
        if is_training:
            conditions = apply_training_noise(conditions, config.NOISE_VH)

        with torch.set_grad_enabled(is_training):
            recon, mu, log_var = model(x, conditions)
            loss, parts = loss_fn(recon, x, mu, log_var)

        if is_training:
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        for key in totals:
            totals[key] += parts[key]
        pbar.set_postfix({k: f"{parts[k]:.4f}" for k in totals})

    n = len(loader)
    return tuple(totals[key] / n for key in ('total', 'recon', 'kld'))


# ==============================================================================
# Training loop
# ==============================================================================
best_val_loss = float('inf')
for epoch in range(config.NUM_EPOCHS):
    print(f"\n--- Epoch {epoch+1}/{config.NUM_EPOCHS} ---")
    loss_fn.beta = cyclical_beta(epoch, config.NUM_EPOCHS, config.BETA_NUM_CYCLES, config.BETA_MAX, config.BETA_RAMP_RATIO)
    current_lr = optimizer.param_groups[0]['lr']
    print(f"LR: {current_lr:.1e}  beta: {loss_fn.beta:.5f}")

    train_total, train_recon, train_kld = run_epoch(model, train_loader, loss_fn, config.DEVICE, True, optimizer)
    print(f"Epoch {epoch+1} | Train | Total: {train_total:.4f} | Recon: {train_recon:.4f} | KLD: {train_kld:.4f}")

    val_total, val_recon, val_kld = run_epoch(model, val_loader, loss_fn, config.DEVICE, False)
    print(f"Epoch {epoch+1} | Valid | Total: {val_total:.4f} | Recon: {val_recon:.4f} | KLD: {val_kld:.4f}")

    scheduler.step(val_recon)

    if val_recon < best_val_loss:
        best_val_loss = val_recon
        print(f"New best validation reconstruction loss. Saving model to {config.SAVE_MODEL_PATH}")
        torch.save(model.state_dict(), config.SAVE_MODEL_PATH)

print("\nTraining complete.")
